# India Multi-Hazard Early Warning System: improved evaluation pipeline

**Branch 1:** Wildfire (existing model remains independent; never retrained or modified).
**Branch 2:** Cyclone -> Rainfall -> Flood (integrated, chronological validation).
**Integration layer:** one dashboard combining available model outputs.

Run cells from top to bottom. Start with `MODE = "FAST"` for 12 representative capital-city points. `MODE = "FULL"` uses 36 state/UT capital points. Neither mode is a district-level or continuous nationwide grid.

**Changes in this improved copy:**
- Classification thresholds are selected on validation data by maximising F2-score (recall-weighted), instead of forcing 98% precision. Validation F2 is also compared with persistence baselines.
- Reports include F2 and PR-AUC alongside accuracy, precision, recall and F1.
- The test set remains held out; threshold selection never uses test labels.
- Existing wildfire model files are only loaded for inference if supplied; this notebook does not retrain or overwrite them.

**Data limitations:**
- Cyclone: NOAA IBTrACS North Indian Ocean.
- Weather/rain/soil moisture: Open-Meteo historical archive (ERA5/ERA5-Land-derived variables); this notebook does not consume NASA GPM or SMAP CSVs as model inputs.
- Flood: Dartmouth Flood Observatory archive when available; large-event centroids are not a complete flood inventory.
- User-uploaded files are copied from `/content` into the matching Drive raw-data folders when their filenames identify the category. Only files explicitly read by a pipeline cell affect that model.
- 98% accuracy is not guaranteed and is not an appropriate stand-alone disaster-warning objective.


In [129]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [130]:
# Cell 1: imports, settings
import os, sys, json, time, math, subprocess, importlib, datetime as dt, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def _ensure(mod, pkg=None):
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg or mod], check=False)
for _m, _p in [("pyarrow", None), ("folium", None), ("openpyxl", None), ("sklearn", "scikit-learn"), ("tqdm", None), ("requests", None)]:
    _ensure(_m, _p)

import numpy as np, pandas as pd, requests
from tqdm.auto import tqdm
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, average_precision_score,
                             precision_recall_curve, mean_absolute_error, mean_squared_error, fbeta_score)

MODE = "FAST"                 # "FAST" = 12 locations, 2008-2021 (~10-20 min).  "FULL" = 36 locations, 2000-2023 (hours, API limits apply)
THRESHOLD_OBJECTIVE = 'F2'    # recall-weighted threshold objective; selected on validation data only
TRAIN_END, VAL_END = 2016, 2018   # train <= 2016, validation 2017-2018, test >= 2019 (chronological)
WIND_THRESH_KT = 34           # cyclone model: storm at tropical-storm strength 24 h ahead
RAIN_THRESH_MM = {6: 10, 12: 20, 24: 35}   # rain classes: mm accumulated in the next 6/12/24 h
FLOOD_RADIUS_KM = 150         # a flood event counts for a location if its centroid is within this distance
USE_PROXY_FLOOD_LABELS = False  # ONLY set True if the observed flood archive cannot be downloaded. Proxy labels are NOT observed floods.

if MODE == "FAST": START_YEAR, END_YEAR, CYC_START = 2008, 2021, 2000
else:              START_YEAR, END_YEAR, CYC_START = 2000, 2023, 1990
FIRST_PRED_YEAR = START_YEAR + 4      # forward-chained predictions (used as inputs to the next model) start here
print("MODE =", MODE, "| years", START_YEAR, "-", END_YEAR)

MODE = FAST | years 2008 - 2021


In [131]:
# Cell 2: Drive (or local folder), folder structure, helpers
try:
    from google.colab import drive
    drive.mount('/content/drive')
    _default_root = '/content/drive/MyDrive/MultiHazardIndia'
except Exception:
    _default_root = str(Path.home() / "MultiHazardIndia")
ROOT = Path(os.environ.get("MH_ROOT", _default_root))
for s in ["raw_data/cyclone", "raw_data/rainfall", "raw_data/weather", "raw_data/flood", "raw_data/soil_moisture",
          "raw_data/elevation", "raw_data/geographic_boundaries", "processed_data", "models", "predictions", "reports", "maps"]:
    (ROOT / s).mkdir(parents=True, exist_ok=True)
RAW, PROC, REPORTS, PRED, MAPS = ROOT/"raw_data", ROOT/"processed_data", ROOT/"reports", ROOT/"predictions", ROOT/"maps"
print("Project folder:", ROOT)

INV_PATH = REPORTS / "dataset_inventory.csv"
inventory = pd.read_csv(INV_PATH).to_dict("records") if INV_PATH.exists() else []
def log(dataset, source, status, path="", date_range="", coverage="", note=""):
    size = Path(path).stat().st_size / 1e6 if path and Path(path).exists() and Path(path).is_file() else 0
    inventory[:] = [r for r in inventory if r.get("dataset") != dataset]
    inventory.append(dict(dataset=dataset, source=source, status=status, path=str(path), date_range=date_range,
                          spatial_coverage=coverage, size_mb=round(size, 2), note=note))
    pd.DataFrame(inventory).to_csv(INV_PATH, index=False)

def download(url, dest, min_bytes=1000):
    """Cached, resumable download. Returns True only if a valid file exists."""
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    if dest.exists() and dest.stat().st_size >= min_bytes:
        print("cached:", dest.name); return True
    pos = part.stat().st_size if part.exists() else 0
    headers = {"Range": f"bytes={pos}-"} if pos else {}
    try:
        with requests.get(url, stream=True, headers=headers, timeout=90) as r:
            if r.status_code not in (200, 206): print("HTTP", r.status_code, url); return False
            total = int(r.headers.get("content-length", 0)) + pos
            with open(part, "ab" if r.status_code == 206 else "wb") as f, tqdm(total=total, initial=pos, unit="B", unit_scale=True, desc=dest.name) as bar:
                for chunk in r.iter_content(1 << 20): f.write(chunk); bar.update(len(chunk))
        if part.stat().st_size < min_bytes: print("file too small, rejecting:", dest.name); return False
        part.rename(dest); return True
    except Exception as e:
        print("download error:", str(e)[:150]); return False

def http_json(url, params, tries=5, pause=1.0):
    for k in range(tries):
        try:
            r = requests.get(url, params=params, timeout=90)
            if r.status_code == 200:
                time.sleep(pause); return r.json()
            txt = r.text[:300].lower()
            if "daily" in txt and "limit" in txt: raise RuntimeError("DAILY_LIMIT")
            if r.status_code == 429 or "limit" in txt: time.sleep(65); continue
            time.sleep(3 * (k + 1))
        except RuntimeError: raise
        except Exception: time.sleep(3 * (k + 1))
    return None

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0; p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def utc_ns(s): return pd.to_datetime(s, utc=True).astype("datetime64[ns, UTC]")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/MultiHazardIndia


In [132]:
# Cell 2A: copy files uploaded through Colab's Files panel into Drive raw_data
# This does NOT assume every uploaded file is used by a model; later cells still
# read only their explicitly configured source files.

import shutil

UPLOAD_ROOT = Path("/content")
UPLOAD_EXTS = {
    ".csv", ".xlsx", ".xls", ".parquet", ".nc", ".h5", ".hdf5",
    ".tif", ".tiff", ".geojson", ".json"
}

def _category_for_upload(filename):
    n = filename.lower()
    if any(k in n for k in ["ibtracs", "cyclone", "storm"]):
        return "cyclone"
    if any(k in n for k in ["flood", "dfo", "indoflood"]):
        return "flood"
    if any(k in n for k in ["firms", "wildfire", "fire", "modis", "viirs"]):
        return "wildfire"
    if any(k in n for k in ["rain", "precip", "gpm", "imd"]):
        return "rainfall"
    if any(k in n for k in ["era5", "weather", "meteo"]):
        return "weather"
    if any(k in n for k in ["smap", "soil"]):
        return "soil_moisture"
    if any(k in n for k in ["dem", "elevation", "srtm", "copernicus"]):
        return "elevation"
    if any(k in n for k in ["boundary", "geoboundary", "gadm", "adm1", "adm2"]):
        return "geographic_boundaries"
    return None

copied_uploads = []
for p in UPLOAD_ROOT.iterdir():
    if not p.is_file() or p.suffix.lower() not in UPLOAD_EXTS:
        continue
    category = _category_for_upload(p.name)
    if category is None:
        continue
    dest_dir = RAW / category
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / p.name
    if not dest.exists():
        shutil.copy2(p, dest)
    copied_uploads.append((p.name, str(dest)))

print("Files copied from Colab /content into Drive raw_data:")
for name, dest in copied_uploads:
    print(" -", name, "->", dest)
if not copied_uploads:
    print("No recognizable uploaded data files found in /content. This is OK if files are already in Drive.")


Files copied from Colab /content into Drive raw_data:
 - floodevents_indofloods (1).csv -> /content/drive/MyDrive/MultiHazardIndia/raw_data/flood/floodevents_indofloods (1).csv
 - ibtracs.ALL.list.v04r01 (2).csv -> /content/drive/MyDrive/MultiHazardIndia/raw_data/cyclone/ibtracs.ALL.list.v04r01 (2).csv
 - metadata_indofloods (2).csv -> /content/drive/MyDrive/MultiHazardIndia/raw_data/flood/metadata_indofloods (2).csv
 - ibtracs.ALL.list.v04r01.csv -> /content/drive/MyDrive/MultiHazardIndia/raw_data/cyclone/ibtracs.ALL.list.v04r01.csv
 - ibtracs.ALL.list.v04r01 (1).csv -> /content/drive/MyDrive/MultiHazardIndia/raw_data/cyclone/ibtracs.ALL.list.v04r01 (1).csv
 - floodevents_indofloods (3).csv -> /content/drive/MyDrive/MultiHazardIndia/raw_data/flood/floodevents_indofloods (3).csv
 - metadata_indofloods.csv -> /content/drive/MyDrive/MultiHazardIndia/raw_data/flood/metadata_indofloods.csv
 - rainfall_statewise_daily_imd (1).csv -> /content/drive/MyDrive/MultiHazardIndia/raw_data/rainfall/

In [133]:
# Cell 3: locations (one per state/UT capital)
ALL_POINTS = {
 "Amaravati (Andhra Pradesh)": (16.51, 80.52), "Itanagar (Arunachal Pradesh)": (27.08, 93.61), "Guwahati (Assam)": (26.14, 91.74),
 "Patna (Bihar)": (25.59, 85.14), "Raipur (Chhattisgarh)": (21.25, 81.63), "Panaji (Goa)": (15.50, 73.83),
 "Gandhinagar (Gujarat)": (23.22, 72.64), "Ambala (Haryana)": (30.38, 76.78), "Shimla (Himachal Pradesh)": (31.10, 77.17),
 "Ranchi (Jharkhand)": (23.34, 85.31), "Bengaluru (Karnataka)": (12.97, 77.59), "Thiruvananthapuram (Kerala)": (8.52, 76.94),
 "Bhopal (Madhya Pradesh)": (23.26, 77.41), "Mumbai (Maharashtra)": (19.08, 72.88), "Imphal (Manipur)": (24.82, 93.94),
 "Shillong (Meghalaya)": (25.58, 91.89), "Aizawl (Mizoram)": (23.73, 92.72), "Kohima (Nagaland)": (25.67, 94.11),
 "Bhubaneswar (Odisha)": (20.30, 85.82), "Ludhiana (Punjab)": (30.90, 75.86), "Jaipur (Rajasthan)": (26.91, 75.79),
 "Gangtok (Sikkim)": (27.33, 88.61), "Chennai (Tamil Nadu)": (13.08, 80.27), "Hyderabad (Telangana)": (17.38, 78.49),
 "Agartala (Tripura)": (23.83, 91.28), "Lucknow (Uttar Pradesh)": (26.85, 80.95), "Dehradun (Uttarakhand)": (30.32, 78.03),
 "Kolkata (West Bengal)": (22.57, 88.36),
 "Port Blair (Andaman & Nicobar, UT)": (11.62, 92.73), "Chandigarh (UT)": (30.73, 76.78), "Daman (Dadra & Nagar Haveli and Daman & Diu, UT)": (20.41, 72.83),
 "Delhi (UT)": (28.61, 77.21), "Srinagar (Jammu & Kashmir, UT)": (34.08, 74.80), "Leh (Ladakh, UT)": (34.15, 77.58),
 "Kavaratti (Lakshadweep, UT)": (10.57, 72.64), "Puducherry (UT)": (11.93, 79.83),
}
FAST_SET = ["Chennai (Tamil Nadu)", "Kolkata (West Bengal)", "Bhubaneswar (Odisha)", "Amaravati (Andhra Pradesh)", "Mumbai (Maharashtra)",
            "Gandhinagar (Gujarat)", "Thiruvananthapuram (Kerala)", "Guwahati (Assam)", "Patna (Bihar)", "Delhi (UT)",
            "Hyderabad (Telangana)", "Lucknow (Uttar Pradesh)"]
POINTS = {k: ALL_POINTS[k] for k in FAST_SET} if MODE == "FAST" else dict(ALL_POINTS)
PT = pd.DataFrame([(i, k, v[0], v[1]) for i, (k, v) in enumerate(POINTS.items())], columns=["pid", "name", "lat", "lon"])
print(len(PT), "locations (28 states + 8 UTs = 36 in FULL mode, capital cities as representative points)")

# optional: official-ish boundaries (non-critical)
try:
    for lvl in ["ADM1", "ADM2"]:
        dest = RAW / "geographic_boundaries" / f"IND_{lvl}.geojson"
        if not dest.exists():
            meta = http_json(f"https://www.geoboundaries.org/api/current/gbOpen/IND/{lvl}/", {}, tries=2)
            if meta and download(meta["gjDownloadURL"], dest, 5000): log(f"boundaries_{lvl}", "geoBoundaries gbOpen", "downloaded", dest, "static", "India")
        else: log(f"boundaries_{lvl}", "geoBoundaries gbOpen", "cached", dest, "static", "India")
except Exception as e:
    print("boundaries skipped (not needed for models):", str(e)[:100])

12 locations (28 states + 8 UTs = 36 in FULL mode, capital cities as representative points)


In [134]:
# Cell 4: CYCLONE data, NOAA IBTrACS (North Indian Ocean)
IBT_URL = "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.NI.list.v04r01.csv"
ibt_raw = RAW / "cyclone" / "ibtracs.NI.list.v04r01.csv"
ok = download(IBT_URL, ibt_raw, 100_000)
assert ok, "IBTrACS download failed. Download the file manually from the NOAA link and put it at: " + str(ibt_raw)
raw = pd.read_csv(ibt_raw, skiprows=[1], low_memory=False)
keep = [c for c in ["SID", "SEASON", "NAME", "ISO_TIME", "LAT", "LON", "USA_WIND", "USA_PRES", "WMO_WIND", "WMO_PRES", "DIST2LAND", "STORM_SPEED", "STORM_DIR"] if c in raw.columns]
cyc = raw[keep].copy()
cyc["ISO_TIME"] = utc_ns(cyc["ISO_TIME"])
for c in ["SEASON", "LAT", "LON", "USA_WIND", "USA_PRES", "WMO_WIND", "WMO_PRES", "DIST2LAND", "STORM_SPEED", "STORM_DIR"]:
    if c in cyc: cyc[c] = pd.to_numeric(cyc[c], errors="coerce")
cyc = cyc.dropna(subset=["ISO_TIME", "LAT", "LON", "SEASON"]).drop_duplicates(["SID", "ISO_TIME"])
cyc["wind_kt"] = cyc["WMO_WIND"].fillna(cyc["USA_WIND"])           # IMD/WMO where reported, else JTWC (different averaging periods, disclosed)
cyc["pres_hpa"] = cyc["WMO_PRES"].fillna(cyc["USA_PRES"])
cyc = cyc[(cyc["SEASON"] >= CYC_START) & cyc["wind_kt"].notna()].sort_values(["SID", "ISO_TIME"]).reset_index(drop=True)
cyc["SEASON"] = cyc["SEASON"].astype(int)
cyc.to_parquet(PROC / "cyclone_tracks_clean.parquet")
log("cyclone_ibtracs", "NOAA IBTrACS v04r01 (NI basin)", "downloaded", ibt_raw, f"{cyc.ISO_TIME.min():%Y}-{cyc.ISO_TIME.max():%Y}", "North Indian Ocean basin (Bay of Bengal + Arabian Sea)")
print(cyc.shape, "| storms:", cyc.SID.nunique(), "| seasons", cyc.SEASON.min(), "-", cyc.SEASON.max())

cached: ibtracs.NI.list.v04r01.csv
(8762, 15) | storms: 266 | seasons 2000 - 2026


In [135]:
# Cell 5: shared modelling utilities
METRICS = []
def make_clf(depth=6, iters=250, leaf=20):
    return HistGradientBoostingClassifier(max_depth=depth, learning_rate=0.06, max_iter=iters, min_samples_leaf=leaf, class_weight="balanced", random_state=42)
def make_reg(depth=6, iters=250):
    return HistGradientBoostingRegressor(max_depth=depth, learning_rate=0.06, max_iter=iters, random_state=42)

def tune_threshold(y_val, p_val):
    """Choose a recall-weighted F2 threshold using validation data only."""
    y_val = np.asarray(y_val).astype(int)
    p_val = np.asarray(p_val, dtype=float)
    if len(y_val) == 0 or len(np.unique(y_val)) < 2:
        return 0.5, False
    prec, rec, thresholds = precision_recall_curve(y_val, p_val)
    if len(thresholds) == 0:
        return 0.5, False
    # precision_recall_curve returns one extra precision/recall point
    # without a corresponding threshold, so use[:-1].
    p, r = prec[:-1], rec[:-1]
    f2 = (5.0 * p * r) / (4.0 * p + r + 1e-12)
    valid = np.isfinite(f2) & np.isfinite(thresholds)
    if not valid.any():
        return 0.5, False
    candidates = np.where(valid)[0]
    best = candidates[np.argmax(f2[valid])]
    return float(thresholds[best]), True

def eval_binary(name, y, prob=None, pred=None, thr=None, note=""):
    y = np.asarray(y).astype(int)
    if pred is None: pred = (np.asarray(prob) >= thr).astype(int)
    pos = int(y.sum())
    row = dict(model=name, n=len(y), positives=pos, threshold=None if thr is None else round(thr, 4),
               accuracy=accuracy_score(y, pred), precision=precision_score(y, pred, zero_division=0),
               recall=recall_score(y, pred, zero_division=0), f1=f1_score(y, pred, zero_division=0), f2=fbeta_score(y, pred, beta=2, zero_division=0),
               pr_auc=(average_precision_score(y, prob) if prob is not None and 0 < pos < len(y) else np.nan), note=note)
    METRICS.append(row)
    print(f"{name:<44} n={len(y):>7} pos={pos:>6} | acc={row['accuracy']:.4f} prec={row['precision']:.4f} rec={row['recall']:.4f} f1={row['f1']:.4f} f2={row['f2']:.4f} PR-AUC={row['pr_auc']:.4f}")
    return row

def forward_chain_oof(df, feats, target, make_model, first_pred_year, proba=True, time_col="time", embargo_days=2):
    """Out-of-sample predictions: year Y is predicted by a model trained ONLY on earlier years (with a small embargo)."""
    out = pd.Series(np.nan, index=df.index)
    years = df[time_col].dt.year
    for y in sorted(years.unique()):
        if y < first_pred_year: continue
        cut = pd.Timestamp(year=int(y), month=1, day=1, tz="UTC") - pd.Timedelta(days=embargo_days)
        tr = df[(df[time_col] < cut) & df[target].notna()]
        te = df[years == y]
        if len(tr) < 300 or te.empty: continue
        yt = tr[target].astype(int) if proba else tr[target]
        if proba and yt.nunique() < 2: continue
        m = make_model().fit(tr[feats], yt)
        out.loc[te.index] = m.predict_proba(te[feats])[:, 1] if proba else m.predict(te[feats])
    return out

In [136]:
# Cell 6: CYCLONE model (independent): classification + wind-speed prediction, 24 h ahead
c = cyc.copy()
fut = c[["SID", "ISO_TIME", "wind_kt"]].copy(); fut["ISO_TIME"] -= pd.Timedelta(hours=24); fut = fut.rename(columns={"wind_kt": "wind_future"})
past = c[["SID", "ISO_TIME", "wind_kt"]].copy(); past["ISO_TIME"] += pd.Timedelta(hours=24); past = past.rename(columns={"wind_kt": "wind_past"})
c = c.merge(fut, on=["SID", "ISO_TIME"], how="left").merge(past, on=["SID", "ISO_TIME"], how="left")   # exact t+24h / t-24h within the same storm, else NaN
c["y_cls"] = np.where(c["wind_future"].notna(), (c["wind_future"] >= WIND_THRESH_KT).astype(float), np.nan)
c["d_wind_24h"] = c["wind_kt"] - c["wind_past"]
c["month"] = c["ISO_TIME"].dt.month
CF = [f for f in ["LAT", "LON", "wind_kt", "pres_hpa", "d_wind_24h", "DIST2LAND", "STORM_SPEED", "STORM_DIR", "month"] if f in c.columns]
lab = c["y_cls"].notna()
tr = c[lab & (c.SEASON <= TRAIN_END)]; va = c[lab & (c.SEASON > TRAIN_END) & (c.SEASON <= VAL_END)]; te = c[lab & (c.SEASON > VAL_END)]
print("cyclone rows train/val/test:", len(tr), len(va), len(te), "| storms in test:", te.SID.nunique())
assert min(len(tr), len(va), len(te)) > 0, "Not enough cyclone seasons for the split."

clf_c = make_clf().fit(tr[CF], tr.y_cls.astype(int))
p_va_c = clf_c.predict_proba(va[CF])[:, 1]
thr_c, reached_c = tune_threshold(va.y_cls, p_va_c)
f2_ml_val = fbeta_score(va.y_cls.astype(int), (p_va_c >= thr_c).astype(int), beta=2, zero_division=0)
f2_base_val = fbeta_score(va.y_cls.astype(int), (va.wind_kt >= WIND_THRESH_KT).astype(int), beta=2, zero_division=0)
print(f"Cyclone validation F2: ML={f2_ml_val:.4f} | persistence={f2_base_val:.4f}")
print(f"cyclone threshold {thr_c:.3f} (selected by maximum validation F2; threshold tuned={reached_c})")
p_te = clf_c.predict_proba(te[CF])[:, 1]
eval_binary("Cyclone classifier TEST", te.y_cls, p_te, thr=thr_c, note=f"wind>={WIND_THRESH_KT}kt at +24h; threshold selected by validation F2={reached_c}")
eval_binary("Cyclone persistence baseline TEST", te.y_cls, pred=(te.wind_kt >= WIND_THRESH_KT).astype(int), note="current wind >= threshold")

reg_c = make_reg().fit(tr[CF], tr.wind_future)
pr = reg_c.predict(te[CF])
print(f"Cyclone wind regression TEST: MAE={mean_absolute_error(te.wind_future, pr):.2f} kt, RMSE={math.sqrt(mean_squared_error(te.wind_future, pr)):.2f} kt | persistence MAE={mean_absolute_error(te.wind_future, te.wind_kt):.2f} kt")
METRICS.append(dict(model="Cyclone wind regressor TEST (MAE kt)", n=len(te), accuracy=np.nan, precision=np.nan, recall=np.nan, f1=np.nan,
                    note=f"MAE={mean_absolute_error(te.wind_future, pr):.2f} kt vs persistence {mean_absolute_error(te.wind_future, te.wind_kt):.2f} kt"))

# final-model probabilities for val/test rows (used by the dashboard)
dash_rows = c[(c.SEASON > TRAIN_END)].copy()
dash_rows["p_cyclone"] = clf_c.predict_proba(dash_rows[CF])[:, 1]
CYC_PRED = dash_rows[["SID", "NAME", "ISO_TIME", "LAT", "LON", "wind_kt", "p_cyclone"]]
CYC_THR = thr_c
CYC_PRED.to_parquet(PRED / "cyclone_predictions_val_test.parquet")

# forward-chained (out-of-sample) cyclone outputs for ALL storm rows -> inputs to the rainfall model
c["p_cyc_oof"] = forward_chain_oof(c, CF, "y_cls", make_clf, CYC_START + 5, proba=True, time_col="ISO_TIME")
c["w24_oof"] = forward_chain_oof(c, CF, "wind_future", make_reg, CYC_START + 5, proba=False, time_col="ISO_TIME")
STORM_STATE = c[["ISO_TIME", "LAT", "LON", "wind_kt", "p_cyc_oof", "w24_oof"]].rename(columns={"LAT": "s_lat", "LON": "s_lon", "wind_kt": "s_wind", "p_cyc_oof": "s_p24", "w24_oof": "s_w24"})
STORM_STATE["ISO_TIME"] = utc_ns(STORM_STATE["ISO_TIME"])
print("cyclone branch done")

cyclone rows train/val/test: 4225 538 2174 | storms in test: 80
Cyclone validation F2: ML=0.9096 | persistence=0.7580
cyclone threshold 0.005 (selected by maximum validation F2; threshold tuned=True)
Cyclone classifier TEST                      n=   2174 pos=  1016 | acc=0.5589 prec=0.5144 rec=1.0000 f1=0.6794 f2=0.8412 PR-AUC=0.8736
Cyclone persistence baseline TEST            n=   2174 pos=  1016 | acc=0.7576 prec=0.7555 rec=0.7116 f1=0.7329 f2=0.7200 PR-AUC=nan
Cyclone wind regression TEST: MAE=9.10 kt, RMSE=13.28 kt | persistence MAE=11.67 kt
cyclone branch done


In [137]:
# Cell 7: WEATHER, RAINFALL, SOIL MOISTURE (ERA5 / ERA5-Land via Open-Meteo, no key) + elevation/slope
VARS = ["temperature_2m", "relative_humidity_2m", "surface_pressure", "wind_speed_10m", "wind_direction_10m", "precipitation", "soil_moisture_0_to_7cm"]
ARCH = "https://archive-api.open-meteo.com/v1/archive"
CACHE = RAW / "weather" / "openmeteo"; CACHE.mkdir(parents=True, exist_ok=True)
HOURLY, ELEV = {}, {}
DAILY_LIMIT_HIT = False

def fetch_hourly(name, lat, lon):
    global DAILY_LIMIT_HIT
    frames = []
    for ya in range(START_YEAR, END_YEAR + 1, 2):
        yb = min(ya + 1, END_YEAR)
        f = CACHE / (("".join(ch if ch.isalnum() else "_" for ch in name)) + f"_{ya}_{yb}.parquet")
        if f.exists(): frames.append(pd.read_parquet(f)); continue
        if DAILY_LIMIT_HIT: continue
        try:
            js = http_json(ARCH, dict(latitude=lat, longitude=lon, start_date=f"{ya}-01-01", end_date=f"{yb}-12-31", hourly=",".join(VARS), timezone="UTC"))
        except RuntimeError:
            DAILY_LIMIT_HIT = True; print("Open-Meteo DAILY limit reached. Re-run this cell tomorrow; cached chunks are kept."); continue
        if not js or "hourly" not in js: print("  failed chunk", name, ya, yb); continue
        d = pd.DataFrame(js["hourly"]); d["time"] = utc_ns(d["time"]); d["elev_api"] = js.get("elevation", np.nan)
        d.to_parquet(f); frames.append(d)
    return pd.concat(frames, ignore_index=True).drop_duplicates("time").sort_values("time").reset_index(drop=True) if frames else None

for _, r in tqdm(PT.iterrows(), total=len(PT), desc="locations"):
    h = fetch_hourly(r["name"], r["lat"], r["lon"])
    if h is not None and len(h) > 1000: HOURLY[r["pid"]] = h
    else: print("NO DATA for", r["name"])
assert HOURLY, "No weather data downloaded. Check internet / API limit and re-run."
PT = PT[PT.pid.isin(HOURLY)].reset_index(drop=True)
log("era5_openmeteo_hourly", "ERA5/ERA5-Land reanalysis via Open-Meteo archive API", f"downloaded for {len(HOURLY)} locations", CACHE,
    f"{START_YEAR}-{END_YEAR}", "point locations only (state/UT capitals), ~9-25 km grid cell each", "temp, humidity, pressure, wind, precipitation, soil moisture 0-7 cm")

# elevation + coarse slope (Open-Meteo elevation API, Copernicus DEM based)
def elev_slope(lat, lon, d=0.05):
    co = [(lat, lon), (lat + d, lon), (lat - d, lon), (lat, lon + d), (lat, lon - d)]
    js = http_json("https://api.open-meteo.com/v1/elevation", dict(latitude=",".join(str(x[0]) for x in co), longitude=",".join(str(x[1]) for x in co)), tries=3, pause=0.3)
    if not js or "elevation" not in js or len(js["elevation"]) < 5: return np.nan, np.nan
    z0, zn, zs, ze, zw = js["elevation"][:5]
    dy = (zn - zs) / (2 * d * 111_200.0); dx = (ze - zw) / (2 * d * 111_200.0 * math.cos(math.radians(lat)))
    return float(z0), float(math.degrees(math.atan(math.hypot(dx, dy))))
for _, r in PT.iterrows():
    z, s = elev_slope(r["lat"], r["lon"])
    ELEV[r["pid"]] = (z if not np.isnan(z) else HOURLY[r["pid"]]["elev_api"].iloc[0], s)
PT["elev"] = PT.pid.map(lambda p: ELEV[p][0]); PT["slope"] = PT.pid.map(lambda p: ELEV[p][1])
log("elevation_slope", "Open-Meteo elevation API (Copernicus DEM 90 m)", "downloaded", "", "static", "point locations; slope is a coarse ~5 km estimate; river proximity NOT computed")
print(PT[["name", "lat", "lon", "elev", "slope"]].round(2).to_string(index=False))

locations:   0%|          | 0/12 [00:00<?, ?it/s]

                       name   lat   lon  elev  slope
       Chennai (Tamil Nadu) 13.08 80.27  10.0   0.07
      Kolkata (West Bengal) 22.57 88.36  12.0   0.02
       Bhubaneswar (Odisha) 20.30 85.82  54.0   0.24
 Amaravati (Andhra Pradesh) 16.51 80.52  21.0   0.05
       Mumbai (Maharashtra) 19.08 72.88   9.0   0.12
      Gandhinagar (Gujarat) 23.22 72.64  81.0   0.06
Thiruvananthapuram (Kerala)  8.52 76.94  26.0   0.25
           Guwahati (Assam) 26.14 91.74  55.0   0.66
              Patna (Bihar) 25.59 85.14  55.0   0.02
                 Delhi (UT) 28.61 77.21 217.0   0.25
      Hyderabad (Telangana) 17.38 78.49 491.0   0.24
    Lucknow (Uttar Pradesh) 26.85 80.95 117.0   0.07


In [138]:

# ============================================================
# CELL 6: CYCLONE MODEL
# Independent classification + 24-hour wind-speed regression
# ============================================================

# Keep a separate copy of cyclone data
c = cyc.copy()

# ------------------------------------------------------------
# 1. CREATE FUTURE AND PAST WIND TARGETS
# ------------------------------------------------------------

fut = c[["SID", "ISO_TIME", "wind_kt"]].copy()
fut["ISO_TIME"] -= pd.Timedelta(hours=24)
fut = fut.rename(columns={"wind_kt": "wind_future"})

past = c[["SID", "ISO_TIME", "wind_kt"]].copy()
past["ISO_TIME"] += pd.Timedelta(hours=24)
past = past.rename(columns={"wind_kt": "wind_past"})

# Match records within the same cyclone
c = (
    c.merge(fut, on=["SID", "ISO_TIME"], how="left")
     .merge(past, on=["SID", "ISO_TIME"], how="left")
)

# ------------------------------------------------------------
# 2. CREATE CLASSIFICATION TARGET AND FEATURES
# ------------------------------------------------------------

c["y_cls"] = np.where(
    c["wind_future"].notna(),
    (c["wind_future"] >= WIND_THRESH_KT).astype(float),
    np.nan
)

c["d_wind_24h"] = c["wind_kt"] - c["wind_past"]
c["month"] = c["ISO_TIME"].dt.month

CF = [
    f for f in [
        "LAT", "LON", "wind_kt", "pres_hpa",
        "d_wind_24h", "DIST2LAND",
        "STORM_SPEED", "STORM_DIR", "month"
    ]
    if f in c.columns
]

# Numeric feature safety
for col in CF:
    c[col] = pd.to_numeric(c[col], errors="coerce")

# ------------------------------------------------------------
# 3. TIME-BASED TRAIN / VALIDATION / TEST SPLIT
# ------------------------------------------------------------

lab = c["y_cls"].notna()

tr_c = c[lab & (c.SEASON <= TRAIN_END)].copy()
va_c = c[
    lab & (c.SEASON > TRAIN_END) &
    (c.SEASON <= VAL_END)
].copy()
te_c = c[lab & (c.SEASON > VAL_END)].copy()

print(
    "Cyclone rows train/val/test:",
    len(tr_c), len(va_c), len(te_c),
    "| storms in test:", te_c.SID.nunique()
)

assert min(len(tr_c), len(va_c), len(te_c)) > 0, (
    "Not enough cyclone seasons for the split."
)

# ------------------------------------------------------------
# 4. CYCLONE CLASSIFICATION
# ------------------------------------------------------------

clf_c = make_clf().fit(
    tr_c[CF],
    tr_c["y_cls"].astype(int)
)

p_va_c = clf_c.predict_proba(va_c[CF])[:, 1]

thr_c, reached_c = tune_threshold(
    va_c["y_cls"],
    p_va_c
)

f2_ml_val = fbeta_score(
    va_c["y_cls"].astype(int),
    (p_va_c >= thr_c).astype(int),
    beta=2,
    zero_division=0
)

f2_base_val = fbeta_score(
    va_c["y_cls"].astype(int),
    (va_c["wind_kt"] >= WIND_THRESH_KT).astype(int),
    beta=2,
    zero_division=0
)

print(
    f"Cyclone validation F2: "
    f"ML={f2_ml_val:.4f} | persistence={f2_base_val:.4f}"
)

print(
    f"Cyclone threshold: {thr_c:.3f} "
    f"(validation F2 tuning successful={reached_c})"
)

p_te_c = clf_c.predict_proba(te_c[CF])[:, 1]

eval_binary(
    "Cyclone classifier TEST",
    te_c["y_cls"],
    p_te_c,
    thr=thr_c,
    note=f"24h cyclone wind threshold={WIND_THRESH_KT} kt"
)

eval_binary(
    "Cyclone persistence baseline TEST",
    te_c["y_cls"],
    pred=(te_c["wind_kt"] >= WIND_THRESH_KT).astype(int),
    note="Current wind >= threshold"
)

# ------------------------------------------------------------
# 5. CYCLONE WIND REGRESSION - CORRECTED
# ------------------------------------------------------------

# Keep only records with valid future wind observations
reg_train = tr_c[
    np.isfinite(tr_c["wind_future"])
].copy()

reg_test = te_c[
    np.isfinite(te_c["wind_future"])
].copy()

assert len(reg_train) > 0, (
    "No valid future wind targets in regression training data."
)

assert len(reg_test) > 0, (
    "No valid future wind targets in regression test data."
)

print("\nWind regression training rows:", len(reg_train))
print("Wind regression testing rows:", len(reg_test))

# Train regression model
reg_c = make_reg()

reg_c.fit(
    reg_train[CF],
    reg_train["wind_future"].astype(float)
)

# Predict future wind speed
wind_pred = reg_c.predict(reg_test[CF])

# Remove any invalid predictions before evaluation
valid = (
    np.isfinite(reg_test["wind_future"].to_numpy()) &
    np.isfinite(wind_pred) &
    np.isfinite(reg_test["wind_kt"].to_numpy())
)

y_wind_true = reg_test["wind_future"].to_numpy()[valid]
y_wind_pred = wind_pred[valid]
y_wind_base = reg_test["wind_kt"].to_numpy()[valid]

assert len(y_wind_true) > 0, (
    "No valid wind predictions available for evaluation."
)

# Calculate regression metrics
wind_mae = mean_absolute_error(
    y_wind_true, y_wind_pred
)

wind_rmse = math.sqrt(
    mean_squared_error(y_wind_true, y_wind_pred)
)

baseline_mae = mean_absolute_error(
    y_wind_true, y_wind_base
)

improvement = (
    (1 - wind_mae / baseline_mae) * 100
    if baseline_mae > 0 else np.nan
)

print("\n" + "=" * 55)
print("CYCLONE WIND REGRESSION TEST RESULTS")
print("=" * 55)

print("Valid test samples:", len(y_wind_true))
print(f"Model MAE: {wind_mae:.2f} knots")
print(f"Model RMSE: {wind_rmse:.2f} knots")
print(f"Persistence baseline MAE: {baseline_mae:.2f} knots")
print(f"Improvement over baseline: {improvement:.2f}%")

# Save regression metrics separately
METRICS.append({
    "model": "Cyclone wind regressor TEST",
    "n": len(y_wind_true),
    "positives": np.nan,
    "accuracy": np.nan,
    "precision": np.nan,
    "recall": np.nan,
    "f1": np.nan,
    "f2": np.nan,
    "pr_auc": np.nan,
    "MAE_kt": wind_mae,
    "RMSE_kt": wind_rmse,
    "baseline_MAE_kt": baseline_mae,
    "note": f"Improvement over baseline={improvement:.2f}%"
})

# ------------------------------------------------------------
# 6. DASHBOARD CYCLONE PREDICTIONS
# ------------------------------------------------------------

dash_rows = c[
    c.SEASON > TRAIN_END
].copy()

dash_rows["p_cyclone"] = clf_c.predict_proba(
    dash_rows[CF]
)[:, 1]

# Future wind prediction for dashboard
dash_rows["predicted_wind_24h_kt"] = reg_c.predict(
    dash_rows[CF]
)

CYC_PRED = dash_rows[
    [
        "SID", "NAME", "ISO_TIME",
        "LAT", "LON", "wind_kt",
        "p_cyclone", "predicted_wind_24h_kt"
    ]
].copy()

CYC_THR = thr_c

CYC_PRED.to_parquet(
    PRED / "cyclone_predictions_val_test.parquet"
)

print("\nCyclone dashboard predictions saved.")
print("Prediction rows:", len(CYC_PRED))

# ------------------------------------------------------------
# 7. FORWARD-CHAINED OOF OUTPUTS
# Used as inputs for rainfall model
# ------------------------------------------------------------

c["p_cyc_oof"] = forward_chain_oof(
    c,
    CF,
    "y_cls",
    make_clf,
    CYC_START + 5,
    proba=True,
    time_col="ISO_TIME"
)

c["w24_oof"] = forward_chain_oof(
    c,
    CF,
    "wind_future",
    make_reg,
    CYC_START + 5,
    proba=False,
    time_col="ISO_TIME"
)

STORM_STATE = c[
    [
        "ISO_TIME", "LAT", "LON",
        "wind_kt", "p_cyc_oof", "w24_oof"
    ]
].rename(
    columns={
        "LAT": "s_lat",
        "LON": "s_lon",
        "wind_kt": "s_wind",
        "p_cyc_oof": "s_p24",
        "w24_oof": "s_w24"
    }
)

STORM_STATE["ISO_TIME"] = utc_ns(
    STORM_STATE["ISO_TIME"]
)

print("\nCyclone classification and wind regression complete.")
print("Cyclone branch done.")


Cyclone rows train/val/test: 4225 538 2174 | storms in test: 80
Cyclone validation F2: ML=0.9096 | persistence=0.7580
Cyclone threshold: 0.005 (validation F2 tuning successful=True)
Cyclone classifier TEST                      n=   2174 pos=  1016 | acc=0.5589 prec=0.5144 rec=1.0000 f1=0.6794 f2=0.8412 PR-AUC=0.8736
Cyclone persistence baseline TEST            n=   2174 pos=  1016 | acc=0.7576 prec=0.7555 rec=0.7116 f1=0.7329 f2=0.7200 PR-AUC=nan

Wind regression training rows: 4225
Wind regression testing rows: 2174

CYCLONE WIND REGRESSION TEST RESULTS
Valid test samples: 2174
Model MAE: 9.10 knots
Model RMSE: 13.28 knots
Persistence baseline MAE: 11.67 knots
Improvement over baseline: 22.04%

Cyclone dashboard predictions saved.
Prediction rows: 3475

Cyclone classification and wind regression complete.
Cyclone branch done.


In [139]:
# Cell 9: RAINFALL model (6 / 12 / 24 h), trained on the cyclone-model outputs + weather features
RAIN_MODELS, RAIN_THR = {}, {}
trm = rain.year <= TRAIN_END; vam = (rain.year > TRAIN_END) & (rain.year <= VAL_END); tem = rain.year > VAL_END
for H in (6, 12, 24):
    tgt = f"y_{H}h"; ok_ = rain[tgt].notna()
    tr, va, te = rain[trm & ok_], rain[vam & ok_], rain[tem & ok_]
    mdl = make_clf(iters=200, leaf=50).fit(tr[RF], tr[tgt].astype(int))
    p_va = mdl.predict_proba(va[RF])[:, 1]
    thr, reached = tune_threshold(va[tgt], p_va)
    f2_ml_val = fbeta_score(va[tgt].astype(int), (p_va >= thr).astype(int), beta=2, zero_division=0)
    f2_base_val = fbeta_score(va[tgt].astype(int), (va[f"pr_{H}h"] >= RAIN_THRESH_MM[H]).astype(int), beta=2, zero_division=0)
    print(f"Rainfall +{H}h validation F2: ML={f2_ml_val:.4f} | persistence={f2_base_val:.4f}")
    pt_ = mdl.predict_proba(te[RF])[:, 1]
    eval_binary(f"Rainfall +{H}h (>= {RAIN_THRESH_MM[H]} mm) TEST", te[tgt], pt_, thr=thr, note=f"validation F2 threshold selected={reached}")
    eval_binary(f"Rainfall +{H}h persistence baseline TEST", te[tgt], pred=(te[f"pr_{H}h"] >= RAIN_THRESH_MM[H]).astype(int), note="past-H-hour rain >= threshold")
    RAIN_MODELS[H], RAIN_THR[H] = mdl, thr
rain["p_rain24"] = np.nan
mask = rain.year > TRAIN_END
rain.loc[mask, "p_rain24"] = RAIN_MODELS[24].predict_proba(rain.loc[mask, RF])[:, 1]
rain.loc[mask, ["pid", "name", "time", "p_rain24"]].to_parquet(PRED / "rainfall24_predictions_val_test.parquet")

# forward-chained (out-of-sample) 24 h rain probability for ALL years -> input to the flood model
rain["p_rain24_oof"] = forward_chain_oof(rain, RF, "y_24h", lambda: make_clf(iters=120, leaf=50), FIRST_PRED_YEAR, proba=True, time_col="time")
print("rainfall branch done")

Rainfall +6h validation F2: ML=0.4768 | persistence=0.3061
Rainfall +6h (>= 10 mm) TEST                 n= 105192 pos=  2272 | acc=0.9393 prec=0.2083 rec=0.6457 f1=0.3150 f2=0.4547 PR-AUC=0.3728
Rainfall +6h persistence baseline TEST       n= 105192 pos=  2272 | acc=0.9705 prec=0.3169 rec=0.3169 f1=0.3169 f2=0.3169 PR-AUC=nan
Rainfall +12h validation F2: ML=0.4879 | persistence=0.3159
Rainfall +12h (>= 20 mm) TEST                n= 105168 pos=  1855 | acc=0.9532 prec=0.2133 rec=0.6162 f1=0.3169 f2=0.4473 PR-AUC=0.3399
Rainfall +12h persistence baseline TEST      n= 105168 pos=  1855 | acc=0.9748 prec=0.2868 rec=0.2868 f1=0.2868 f2=0.2868 PR-AUC=nan
Rainfall +24h validation F2: ML=0.4873 | persistence=0.3230
Rainfall +24h (>= 35 mm) TEST                n= 105120 pos=  1994 | acc=0.9542 prec=0.2208 rec=0.5587 f1=0.3165 f2=0.4278 PR-AUC=0.3195
Rainfall +24h persistence baseline TEST      n= 105120 pos=  1994 | acc=0.9716 prec=0.2508 rec=0.2508 f1=0.2508 f2=0.2508 PR-AUC=nan
rainfall branc

In [140]:

# Cell 10: FLOOD labels using INDOFLOODS events + station metadata

FLOOD_DIR = RAW / "flood"

EVENT_FILE = FLOOD_DIR / "floodevents_indofloods.csv"
META_FILE = FLOOD_DIR / "metadata_indofloods.csv"

assert EVENT_FILE.exists(), f"Missing file: {EVENT_FILE}"
assert META_FILE.exists(), f"Missing file: {META_FILE}"

# Read datasets
events = pd.read_csv(EVENT_FILE)
meta = pd.read_csv(META_FILE)

# Clean column names and identifiers
events.columns = events.columns.str.strip()
meta.columns = meta.columns.str.strip()

events["EventID"] = events["EventID"].astype(str).str.strip()
meta["GaugeID"] = meta["GaugeID"].astype(str).str.strip()

# EventID example:
# INDOFLOODS-gauge-1010-1
# Corresponding GaugeID:
# INDOFLOODS-gauge-1010

events["GaugeID"] = events["EventID"].str.replace(
    r"-\d+$", "", regex=True
)

# Prepare station coordinates
meta["Latitude"] = pd.to_numeric(meta["Latitude"], errors="coerce")
meta["Longitude"] = pd.to_numeric(meta["Longitude"], errors="coerce")

# Match every flood event to its gauge station
EV = events.merge(
    meta[["GaugeID", "Latitude", "Longitude", "Station", "State", "Basin"]],
    on="GaugeID",
    how="left",
    validate="many_to_one"
)

# Convert to the format expected by the existing flood model
EV = EV.rename(columns={
    "Latitude": "lat",
    "Longitude": "lon",
    "Start Date": "began",
    "End Date": "ended"
})

EV["began"] = pd.to_datetime(EV["began"], errors="coerce")
EV["ended"] = pd.to_datetime(EV["ended"], errors="coerce")

EV["lat"] = pd.to_numeric(EV["lat"], errors="coerce")
EV["lon"] = pd.to_numeric(EV["lon"], errors="coerce")

# Keep records with valid event dates and station coordinates
matched_before = len(EV)
EV = EV.dropna(subset=["began", "lat", "lon"]).copy()

# Keep events within the notebook's modelling period
EV = EV[
    (EV["began"].dt.year >= START_YEAR) &
    (EV["began"].dt.year <= END_YEAR)
].copy()

EV = EV.drop_duplicates(subset=["EventID"]).reset_index(drop=True)

unmatched = events["GaugeID"].nunique() - events[
    "GaugeID"
].isin(meta["GaugeID"]).sum()

print("INDOFLOODS flood dataset loaded successfully")
print("Original events:", len(events))
print("Events with valid station coordinates:", matched_before)
print("Events in modelling period:", len(EV))
print("Unique gauge stations matched:", EV["GaugeID"].nunique())
print("Unmatched event records:", events["GaugeID"].isin(meta["GaugeID"]).eq(False).sum())

print("\nFlood events by state:")
print(EV["State"].value_counts().head(15))

print("\nSample matched events:")
display(EV[[
    "EventID", "GaugeID", "began", "ended",
    "lat", "lon", "Station", "State", "Basin"
]].head())

# Register dataset in project inventory
log(
    "flood_events_indofloods",
    "INDOFLOODS flood events joined with gauge metadata",
    f"{len(EV)} matched events",
    EVENT_FILE,
    f"{START_YEAR}-{END_YEAR}",
    "214 gauge metadata records; event locations from station coordinates"
)

# GloFAS river discharge (modelled, not observed)
DISCH = {}

for _, r in tqdm(PT.iterrows(), total=len(PT), desc="river discharge"):
    f = CACHE / (
        "disch_" +
        "".join(ch if ch.isalnum() else "_" for ch in r["name"]) +
        f"_{START_YEAR}_{END_YEAR}.parquet"
    )

    if f.exists():
        DISCH[r["pid"]] = pd.read_parquet(f)
        continue

    try:
        js = http_json(
            "https://flood-api.open-meteo.com/v1/flood",
            dict(
                latitude=r["lat"],
                longitude=r["lon"],
                daily="river_discharge",
                start_date=f"{START_YEAR}-01-01",
                end_date=f"{END_YEAR}-12-31"
            ),
            tries=3,
            pause=0.5
        )
    except RuntimeError:
        js = None

    if js and "daily" in js:
        d = pd.DataFrame(js["daily"])
        d["date"] = pd.to_datetime(d["time"]).dt.normalize()
        d = d[["date", "river_discharge"]]
        d.to_parquet(f)
        DISCH[r["pid"]] = d

print("\nDischarge available for", len(DISCH), "of", len(PT), "locations")
if DISCH:
    log(
        "river_discharge_glofas",
        "GloFAS via Open-Meteo flood API (modelled)",
        f"{len(DISCH)} locations",
        CACHE,
        f"{START_YEAR}-{END_YEAR}",
        "point locations"
    )


INDOFLOODS flood dataset loaded successfully
Original events: 4548
Events with valid station coordinates: 4548
Events in modelling period: 1472
Unique gauge stations matched: 146
Unmatched event records: 0

Flood events by state:
State
Karnataka                                   479
West Bengal                                 242
Kerala                                      145
Odisha                                      126
Madhya Pradesh                              121
Maharashtra                                 119
Tamil Nadu                                   69
Andhra Pradesh                               64
Telangana                                    57
Chhattisgarh                                 20
Gujarat                                      18
Jharkhand                                     4
Dadra and Nagar Haveli and Daman and Diu      4
Rajasthan                                     2
Tripura                                       2
Name: count, dtype: int64

Sample matched ev

,EventID,GaugeID,began,ended,lat,lon,Station,State,Basin
0,INDOFLOODS-gauge-1010-1,INDOFLOODS-gauge-1010,2010-07-21,2010-07-21,26.46,89.86,L.R.P. Crossing,West Bengal,NaN
1,INDOFLOODS-gauge-1010-2,INDOFLOODS-gauge-1010,2016-07-23,2016-07-23,26.46,89.86,L.R.P. Crossing,West Bengal,NaN
2,INDOFLOODS-gauge-1010-3,INDOFLOODS-gauge-1010,2016-07-26,2016-07-26,26.46,89.86,L.R.P. Crossing,West Bengal,NaN
3,INDOFLOODS-gauge-1010-4,INDOFLOODS-gauge-1010,2017-08-11,2017-08-13,26.46,89.86,L.R.P. Crossing,West Bengal,NaN
4,INDOFLOODS-gauge-1012-1,INDOFLOODS-gauge-1012,2010-07-21,2010-07-21,26.47,89.79,L.R.P. Crossing,West Bengal,NaN


river discharge:   0%|          | 0/12 [00:00<?, ?it/s]


Discharge available for 12 of 12 locations


In [141]:
# FIX: Ensure elevation and slope columns exist in PT

import numpy as np

if "elev" not in PT.columns:
    PT["elev"] = np.nan

if "slope" not in PT.columns:
    PT["slope"] = np.nan

# Recover elevation from cached weather data where possible
for pid in PT["pid"]:
    if pid in HOURLY and PT.loc[PT["pid"] == pid, "elev"].isna().any():
        h = HOURLY[pid]
        if "elev_api" in h.columns:
            PT.loc[PT["pid"] == pid, "elev"] = h["elev_api"].iloc[0]

print("PT columns:", PT.columns.tolist())
print("Missing elevation:", PT["elev"].isna().sum())
print("Missing slope:", PT["slope"].isna().sum())

display(PT.head())

PT columns: ['pid', 'name', 'lat', 'lon', 'elev', 'slope']
Missing elevation: 0
Missing slope: 0


,pid,name,lat,lon,elev,slope
0,0,Chennai (Tamil Nadu),13.08,80.27,10.0,0.073434
1,1,Kolkata (West Bengal),22.57,88.36,12.0,0.022906
2,2,Bhubaneswar (Odisha),20.30,85.82,54.0,0.243091
3,3,Amaravati (Andhra Pradesh),16.51,80.52,21.0,0.047677
4,4,Mumbai (Maharashtra),19.08,72.88,9.0,0.119631


In [142]:
# Cell 11: FLOOD dataset + FLOOD model (inputs: rainfall forecast, soil moisture, cyclone outputs, terrain, discharge)
FLOOD_OK = False
if (EV is not None and len(EV) > 0) or USE_PROXY_FLOOD_LABELS:
    daily_parts = []
    for pid in PT.pid:
        h = HOURLY[pid]; h = h.assign(date=h["time"].dt.tz_convert("UTC").dt.tz_localize(None).dt.normalize())
        g = h.groupby("date").agg(pr_1d=("precipitation", "sum"), pr_hmax=("precipitation", "max"), sm=("soil_moisture_0_to_7cm", "mean"), temp_d=("temperature_2m", "mean")).reset_index()
        g["pid"] = pid
        for k in (3, 7, 14, 30): g[f"pr_{k}d"] = g["pr_1d"].rolling(k, min_periods=k).sum()
        g["sm_7d"] = g["sm"].rolling(7, min_periods=7).mean()
        if pid in DISCH:
            g = g.merge(DISCH[pid], on="date", how="left")
            g["q_3d"] = g["river_discharge"].rolling(3, min_periods=3).mean(); g["q_ratio_30d"] = g["river_discharge"] / g["river_discharge"].rolling(30, min_periods=30).mean()
        else: g["river_discharge"] = g["q_3d"] = g["q_ratio_30d"] = np.nan
        daily_parts.append(g)
    fl = pd.concat(daily_parts, ignore_index=True)
    r2 = rain.assign(date=rain["time"].dt.tz_localize(None).dt.normalize())
    agg = r2.groupby(["pid", "date"]).agg(rain_p24_oof=("p_rain24_oof", "max"), storm_dist_min=("storm_dist_km", "min"), storm_p24_max=("storm_p24", "max"), storm_wind_max=("storm_wind_now", "max")).reset_index()
    fl = fl.merge(agg, on=["pid", "date"], how="left").merge(PT[["pid", "name", "lat", "lon", "elev", "slope"]], on="pid", how="left")
    fl["year"] = fl["date"].dt.year; fl["mon_sin"] = np.sin(2 * np.pi * fl["date"].dt.month / 12); fl["mon_cos"] = np.cos(2 * np.pi * fl["date"].dt.month / 12)

    lab = pd.Series(False, index=fl.index); label_kind = ""
    if EV is not None and len(EV) > 0:
        label_kind = "observed (DFO): a flood event BEGINS within the next 3 days within %d km" % FLOOD_RADIUS_KM
        for pid, g in fl.groupby("pid"):
            la, lo = PT.loc[PT.pid == pid, ["lat", "lon"]].iloc[0]
            near = EV[haversine(la, lo, EV["lat"].values, EV["lon"].values) <= FLOOD_RADIUS_KM]
            onset = near["began"].dt.normalize().values
            hit = np.zeros(len(g), dtype=bool)
            for k in (1, 2, 3): hit |= np.isin((g["date"] + pd.Timedelta(days=k)).values, onset)
            lab.loc[g.index] = hit
    else:
        label_kind = "PROXY (NOT observed): modelled GloFAS discharge in next 3 days above the location's 95th percentile (computed on training years only)"
        for pid, g in fl.groupby("pid"):
            q = g["river_discharge"]; q95 = q[g["year"] <= TRAIN_END].quantile(0.95)
            nxt = pd.concat([q.shift(-k) for k in (1, 2, 3)], axis=1).max(axis=1, skipna=False)
            lab.loc[g.index] = (nxt >= q95) & nxt.notna()
    fl["y_flood"] = lab.astype(int)
    FFEAT = ["lat", "lon", "elev", "slope", "pr_1d", "pr_hmax", "pr_3d", "pr_7d", "pr_14d", "pr_30d", "sm", "sm_7d", "rain_p24_oof", "storm_dist_min", "storm_p24_max", "storm_wind_max", "mon_sin", "mon_cos"]
    if label_kind.startswith("observed"): FFEAT += ["river_discharge", "q_3d", "q_ratio_30d"]     # discharge excluded for proxy labels (circular)
    fl = fl[(fl.year >= FIRST_PRED_YEAR)]
    fl = fl[fl["pr_30d"].notna()]
    fl.to_parquet(PROC / "flood_ml_dataset.parquet")
    trf, vaf, tef = fl[fl.year <= TRAIN_END], fl[(fl.year > TRAIN_END) & (fl.year <= VAL_END)], fl[fl.year > VAL_END]
    print("flood label:", label_kind)
    print("positives train/val/test:", int(trf.y_flood.sum()), int(vaf.y_flood.sum()), int(tef.y_flood.sum()), "| rows:", len(trf), len(vaf), len(tef))
    if trf.y_flood.sum() >= 10 and tef.y_flood.sum() > 0:
        fm = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200, min_samples_leaf=40, class_weight="balanced", random_state=42).fit(trf[FFEAT], trf.y_flood)
        thr_f, reached_f = tune_threshold(vaf.y_flood, fm.predict_proba(vaf[FFEAT])[:, 1])
        pf = fm.predict_proba(tef[FFEAT])[:, 1]
        eval_binary("Flood onset (next 3 days) TEST", tef.y_flood, pf, thr=thr_f, note=f"{label_kind[:60]}; validation F2 threshold selected={reached_f}")
        eval_binary("Flood persistence baseline TEST (rain_7d top 5%)", tef.y_flood, pred=(tef["pr_7d"] >= trf["pr_7d"].quantile(0.95)).astype(int), note="heavy past-week rain")
        fl["p_flood"] = np.nan; vt = fl.year > TRAIN_END
        fl.loc[vt, "p_flood"] = fm.predict_proba(fl.loc[vt, FFEAT])[:, 1]
        FLOOD_PRED, FLOOD_THR, FLOOD_OK = fl.loc[vt, ["pid", "name", "date", "p_flood", "y_flood"]], thr_f, True
        FLOOD_PRED.to_parquet(PRED / "flood_predictions_val_test.parquet")
    else:
        print("Too few flood positives in train/test to train or evaluate a flood model honestly. Add more locations (MODE='FULL') or a longer period.")
else:
    print("Flood model skipped: no observed labels.")

flood label: observed (DFO): a flood event BEGINS within the next 3 days within 150 km
positives train/val/test: 228 116 172 | rows: 21924 8760 13152
Flood onset (next 3 days) TEST               n=  13152 pos=   172 | acc=0.8948 prec=0.0395 rec=0.3023 f1=0.0699 f2=0.1297 PR-AUC=0.0316
Flood persistence baseline TEST (rain_7d top 5%) n=  13152 pos=   172 | acc=0.9312 prec=0.0271 rec=0.1221 f1=0.0444 f2=0.0718 PR-AUC=nan


In [143]:
# Cell 12: WILDFIRE branch (independent; the existing model is only READ, never modified)
WF_DIR = ROOT / "models" / "wildfire"; WF_DIR.mkdir(parents=True, exist_ok=True)
# (a) prepare compatible daily weather inputs for you to map onto your wildfire model's features (no model involved)
if HOURLY:
    wparts = []
    for pid in PT.pid:
        h = HOURLY[pid]; h = h.assign(date=h["time"].dt.tz_localize(None).dt.normalize())
        g = h.groupby("date").agg(tmax=("temperature_2m", "max"), rh_min=("relative_humidity_2m", "min"), wind_max=("wind_speed_10m", "max"), precip=("precipitation", "sum"), soil_m=("soil_moisture_0_to_7cm", "mean")).reset_index()
        g["precip_7d"] = g["precip"].rolling(7, min_periods=7).sum(); g["precip_30d"] = g["precip"].rolling(30, min_periods=30).sum()
        g["name"] = PT.loc[PT.pid == pid, "name"].iloc[0]; g["lat"] = PT.loc[PT.pid == pid, "lat"].iloc[0]; g["lon"] = PT.loc[PT.pid == pid, "lon"].iloc[0]
        wparts.append(g)
    wf_inputs = pd.concat(wparts, ignore_index=True)
    out = PROC / "wildfire_input_candidates.parquet"
    if not out.exists(): wf_inputs.to_parquet(out)
    print("Wildfire candidate inputs saved:", out, "(map these columns to your model's expected features)")
# (b) run YOUR existing model if its files and a ready-made input table are present
model_files = [p for p in list(WF_DIR.glob("*")) if p.suffix.lower() in (".joblib", ".pkl", ".pickle", ".keras", ".h5", ".pt")]
wf_in = WF_DIR / "wildfire_input.csv"     # columns: name, date, + the exact features your model was trained on, in the right order/scaling
WILDFIRE_PRED = None
if model_files and wf_in.exists():
    mf = model_files[0]; X = pd.read_csv(wf_in)
    feat_cols = [c_ for c_ in X.columns if c_ not in ("name", "date")]
    try:
        if mf.suffix.lower() in (".joblib", ".pkl", ".pickle"):
            import joblib; mdl_w = joblib.load(mf)
            pw = mdl_w.predict_proba(X[feat_cols])[:, 1] if hasattr(mdl_w, "predict_proba") else np.asarray(mdl_w.predict(X[feat_cols])).ravel()
        else:
            import tensorflow as tf; mdl_w = tf.keras.models.load_model(mf, compile=False); pw = np.asarray(mdl_w.predict(X[feat_cols].values)).ravel()
        WILDFIRE_PRED = pd.DataFrame({"name": X["name"], "date": pd.to_datetime(X["date"]).dt.normalize(), "p_wildfire": pw})
        outp = PRED / "wildfire_predictions.csv"
        if outp.exists(): outp = PRED / f"wildfire_predictions_{dt.datetime.now():%Y%m%d_%H%M%S}.csv"      # never overwrite
        WILDFIRE_PRED.to_csv(outp, index=False); print("Wildfire predictions saved:", outp)
    except Exception as e:
        print("Could not run the wildfire model (left untouched):", str(e)[:200])
else:
    print("Wildfire model/input not found in", WF_DIR, "\n -> upload your existing model file and a 'wildfire_input.csv' there. Dashboard will show wildfire as N/A until then (no fake values).")

Wildfire candidate inputs saved: /content/drive/MyDrive/MultiHazardIndia/processed_data/wildfire_input_candidates.parquet (map these columns to your model's expected features)
Wildfire model/input not found in /content/drive/MyDrive/MultiHazardIndia/models/wildfire 
 -> upload your existing model file and a 'wildfire_input.csv' there. Dashboard will show wildfire as N/A until then (no fake values).


In [144]:
# Cell 13: INTEGRATED DASHBOARD (replay of one date from the held-out test period)
import folium
try:
    from IPython.display import display
except Exception:
    display = lambda *_a, **_k: None
ORDER = {"N/A": -1, "Low": 0, "Moderate": 1, "High": 2, "Severe": 3}
def level(p, thr):
    if p is None or pd.isna(p): return "N/A"
    if p < 0.5 * thr: return "Low"
    if p < thr: return "Moderate"
    if p < (1 + thr) / 2: return "High"
    return "Severe"

rp = pd.read_parquet(PRED / "rainfall24_predictions_val_test.parquet"); rp["date"] = rp["time"].dt.tz_localize(None).dt.normalize()
rain_d = rp[rp.date.dt.year > VAL_END].groupby(["name", "date"])["p_rain24"].max().reset_index()
if FLOOD_OK: cand = FLOOD_PRED[FLOOD_PRED.date.dt.year > VAL_END].groupby("date")["p_flood"].max()
else:        cand = rain_d.groupby("date")["p_rain24"].max()
DASH_DATE = pd.Timestamp(os.environ.get("DASHBOARD_DATE", "")) if os.environ.get("DASHBOARD_DATE") else cand.idxmax()
print("Dashboard date (replay of test period, highest-risk day):", DASH_DATE.date())

cp = CYC_PRED.copy(); cp["date"] = cp["ISO_TIME"].dt.tz_localize(None).dt.normalize(); cps = cp[cp.date == DASH_DATE]
rows = []
for _, p in PT.iterrows():
    r_ = rain_d[(rain_d.name == p["name"]) & (rain_d.date == DASH_DATE)]["p_rain24"]
    f_ = FLOOD_PRED[(FLOOD_PRED.name == p["name"]) & (FLOOD_PRED.date == DASH_DATE)]["p_flood"] if FLOOD_OK else pd.Series(dtype=float)
    if len(cps):
        d_ = haversine(p["lat"], p["lon"], cps["LAT"].values, cps["LON"].values); near = cps[d_ <= 500]
        pc = float(near["p_cyclone"].max()) if len(near) else 0.0     # 0 = no storm within 500 km that day
    else: pc = 0.0
    w_ = np.nan
    if WILDFIRE_PRED is not None:
        ww = WILDFIRE_PRED[(WILDFIRE_PRED.name == p["name"]) & (WILDFIRE_PRED.date == DASH_DATE)]["p_wildfire"]
        w_ = float(ww.iloc[0]) if len(ww) else np.nan
    row = dict(location=p["name"], lat=p["lat"], lon=p["lon"], p_cyclone=pc, p_rain24=float(r_.iloc[0]) if len(r_) else np.nan,
               p_flood=float(f_.iloc[0]) if len(f_) else np.nan, p_wildfire=w_)
    row["lvl_cyclone"] = level(pc, CYC_THR); row["lvl_rain"] = level(row["p_rain24"], RAIN_THR[24])
    row["lvl_flood"] = level(row["p_flood"], FLOOD_THR) if FLOOD_OK else "N/A"; row["lvl_wildfire"] = level(w_, 0.5)
    row["overall"] = max([row[k] for k in ("lvl_cyclone", "lvl_rain", "lvl_flood", "lvl_wildfire")], key=lambda x: ORDER[x])
    rows.append(row)
dash = pd.DataFrame(rows); dash.insert(0, "date", DASH_DATE.date())
dash.to_csv(PRED / f"dashboard_{DASH_DATE:%Y%m%d}.csv", index=False)
print(dash[["location", "p_cyclone", "lvl_cyclone", "p_rain24", "lvl_rain", "p_flood", "lvl_flood", "lvl_wildfire", "overall"]].round(3).to_string(index=False))
cnt = dash.overall.value_counts().to_dict()
worst = dash.sort_values("overall", key=lambda s: s.map(ORDER), ascending=False).head(5).location.tolist()
print("\nOVERALL WARNING SUMMARY:", {k: cnt.get(k, 0) for k in ["Severe", "High", "Moderate", "Low", "N/A"]}, "\nHighest-risk locations:", worst)
colors = {"Low": "green", "Moderate": "gold", "High": "orange", "Severe": "red", "N/A": "gray"}
mp = folium.Map(location=[22.5, 80.0], zoom_start=5)
for _, r in dash.iterrows():
    pop = (f"<b>{r.location}</b><br>Cyclone: {r.lvl_cyclone} ({r.p_cyclone:.2f})<br>Rain 24h: {r.lvl_rain} ({r.p_rain24:.2f})"
           f"<br>Flood: {r.lvl_flood} ({r.p_flood:.2f})<br>Wildfire: {r.lvl_wildfire}<br><b>Overall: {r.overall}</b>")
    folium.CircleMarker([r.lat, r.lon], radius=9, color=colors[r.overall], fill=True, fill_opacity=0.8, popup=folium.Popup(pop, max_width=260)).add_to(mp)
map_path = MAPS / f"dashboard_{DASH_DATE:%Y%m%d}.html"; mp.save(str(map_path)); print("Map saved:", map_path)
try: display(mp)
except Exception: pass

Dashboard date (replay of test period, highest-risk day): 2021-05-25
                   location  p_cyclone lvl_cyclone  p_rain24 lvl_rain  p_flood lvl_flood lvl_wildfire overall
       Chennai (Tamil Nadu)      0.000         Low     0.094      Low    0.003       Low          N/A     Low
      Kolkata (West Bengal)      0.994      Severe     0.908   Severe    0.055       Low          N/A  Severe
       Bhubaneswar (Odisha)      0.994      Severe     0.988   Severe    0.050       Low          N/A  Severe
 Amaravati (Andhra Pradesh)      0.000         Low     0.010      Low    0.003       Low          N/A     Low
       Mumbai (Maharashtra)      0.000         Low     0.043      Low    0.014       Low          N/A     Low
      Gandhinagar (Gujarat)      0.000         Low     0.006      Low    0.002       Low          N/A     Low
Thiruvananthapuram (Kerala)      0.000         Low     0.466 Moderate    0.971    Severe          N/A  Severe
           Guwahati (Assam)      0.000         Low 

In [145]:
# Cell 14: final report (metrics + honest limitations)
met = pd.DataFrame(METRICS)
met["accuracy>=98%"] = met["accuracy"] >= 0.98; met["precision>=98%"] = met["precision"] >= 0.98
met.to_csv(REPORTS / "model_metrics.csv", index=False)
print(met[["model", "n", "positives", "accuracy", "precision", "recall", "f1", "f2", "pr_auc", "accuracy>=98%", "precision>=98%"]].round(4).to_string(index=False))
lim = f"""# Multi-hazard run report ({dt.date.today()}), MODE={MODE}
## Data actually used
- Cyclone: NOAA IBTrACS NI basin ({CYC_START}+). Wind = IMD/WMO where reported else JTWC (different averaging periods).
- Weather, rainfall, soil moisture: ERA5/ERA5-Land reanalysis through the Open-Meteo API (NOT NASA IMERG / SMAP).
- Flood labels: {'Dartmouth Flood Observatory observed events' if EV is not None and len(EV) else 'NOT AVAILABLE or PROXY (see notebook output)'}.
- Terrain: Open-Meteo elevation (Copernicus DEM); slope is coarse (~5 km); river proximity not computed; GloFAS modelled discharge used as a hydrology input.
## Limitations (state these in your submission)
- Coverage is {len(PT)} point locations ({'state/UT capitals' if MODE=='FULL' else 'a 12-location subset'}), NOT a continuous nationwide grid or district-level map.
- Period {START_YEAR}-{END_YEAR}; train <= {TRAIN_END}, validation {TRAIN_END+1}-{VAL_END}, test >= {VAL_END+1}. Dashboard is a replay of a test-period day, not a live forecast.
- Flood events in the archive are large floods with centroid locations; matching radius {FLOOD_RADIUS_KM} km. Few positives, so flood metrics are noisy.
- Metrics are in model_metrics.csv, including F2 and PR-AUC. Thresholds are selected on validation data by maximising F2; 98% accuracy/precision are not guaranteed.
- Wildfire: existing model was not retrained or modified; predictions appear only if its files and inputs were provided.
"""
(REPORTS / "run_report.md").write_text(lim); print("\nSaved reports to", REPORTS)

                                           model      n  positives  accuracy  precision  recall     f1     f2  pr_auc  accuracy>=98%  precision>=98%
                         Cyclone classifier TEST   2174     1016.0    0.5589     0.5144  1.0000 0.6794 0.8412  0.8736          False           False
               Cyclone persistence baseline TEST   2174     1016.0    0.7576     0.7555  0.7116 0.7329 0.7200     NaN          False           False
            Cyclone wind regressor TEST (MAE kt)   2174        NaN       NaN        NaN     NaN    NaN    NaN     NaN          False           False
                         Cyclone classifier TEST   2174     1016.0    0.5589     0.5144  1.0000 0.6794 0.8412  0.8736          False           False
               Cyclone persistence baseline TEST   2174     1016.0    0.7576     0.7555  0.7116 0.7329 0.7200     NaN          False           False
                     Cyclone wind regressor TEST   2174        NaN       NaN        NaN     NaN    NaN    

In [148]:
print("CYCLONE WIND REGRESSION DIAGNOSTIC")

print("MAE:", wind_mae)
print("RMSE:", wind_rmse)
print("Baseline MAE:", baseline_mae)

print("\nSaved regression records:")

for m in METRICS:
    if "wind regressor" in m.get("model", "").lower():
        print(m)

CYCLONE WIND REGRESSION DIAGNOSTIC
MAE: 9.099240571603856
RMSE: 13.27898096104763
Baseline MAE: 11.671573137074517

Saved regression records:
{'model': 'Cyclone wind regressor TEST (MAE kt)', 'n': 2174, 'accuracy': nan, 'precision': nan, 'recall': nan, 'f1': nan, 'note': 'MAE=9.10 kt vs persistence 11.67 kt'}
{'model': 'Cyclone wind regressor TEST', 'n': 2174, 'positives': nan, 'accuracy': nan, 'precision': nan, 'recall': nan, 'f1': nan, 'f2': nan, 'pr_auc': nan, 'MAE_kt': 9.099240571603856, 'RMSE_kt': 13.27898096104763, 'baseline_MAE_kt': 11.671573137074517, 'note': 'Improvement over baseline=22.04%'}


In [149]:
# Remove obsolete wind regression record
METRICS[:] = [
    m for m in METRICS
    if m.get("model") != "Cyclone wind regressor TEST (MAE kt)"
]

# Keep only the latest record for each model
latest = {}
for m in METRICS:
    latest[m["model"]] = m

METRICS[:] = list(latest.values())

# Ensure regression columns exist
met = pd.DataFrame(METRICS)

for col in ["MAE_kt", "RMSE_kt", "baseline_MAE_kt"]:
    if col not in met.columns:
        met[col] = np.nan

# Display full report
display(met[[
    "model", "n", "accuracy", "precision",
    "recall", "f1", "f2", "pr_auc",
    "MAE_kt", "RMSE_kt", "baseline_MAE_kt"
]].round(4))

# Save corrected report
met.to_csv(REPORTS / "model_metrics.csv", index=False)

print("Clean report saved successfully.")

,model,n,accuracy,precision,recall,f1,f2,pr_auc,MAE_kt,RMSE_kt,baseline_MAE_kt
0,Cyclone classifier TEST,2174,0.5589,0.5144,1.0000,0.6794,0.8412,0.8736,NaN,NaN,NaN
1,Cyclone persistence baseline TEST,2174,0.7576,0.7555,0.7116,0.7329,0.7200,NaN,NaN,NaN,NaN
2,Cyclone wind regressor TEST,2174,NaN,NaN,NaN,NaN,NaN,NaN,9.0992,13.279,11.6716
3,Rainfall +6h (>= 10 mm) TEST,105192,0.9393,0.2083,0.6457,0.3150,0.4547,0.3728,NaN,NaN,NaN
4,Rainfall +6h persistence baseline TEST,105192,0.9705,0.3169,0.3169,0.3169,0.3169,NaN,NaN,NaN,NaN
5,Rainfall +12h (>= 20 mm) TEST,105168,0.9532,0.2133,0.6162,0.3169,0.4473,0.3399,NaN,NaN,NaN
6,Rainfall +12h persistence baseline TEST,105168,0.9748,0.2868,0.2868,0.2868,0.2868,NaN,NaN,NaN,NaN
7,Rainfall +24h (>= 35 mm) TEST,105120,0.9542,0.2208,0.5587,0.3165,0.4278,0.3195,NaN,NaN,NaN
8,Rainfall +24h persistence baseline TEST,105120,0.9716,0.2508,0.2508,0.2508,0.2508,NaN,NaN,NaN,NaN
9,Flood onset (next 3 days) TEST,13152,0.8948,0.0395,0.3023,0.0699,0.1297,0.0316,NaN,NaN,NaN


Clean report saved successfully.


In [146]:
import os
import pandas as pd

folder = "/content/drive/MyDrive/MultiHazardIndia/raw_data/flood"

for filename in [
    "floodevents_indofloods.csv",
    "metadata_indofloods.csv",
    "precipitation_variables_indofloods.csv",
    "catchment_characteristics_indofloods.csv"
]:
    path = os.path.join(folder, filename)

    print("\n" + "="*60)
    print(filename)

    if os.path.exists(path):
        df = pd.read_csv(path)
        print("FOUND")
        print("Rows and columns:", df.shape)
        print("Column names:", df.columns.tolist())
        display(df.head(3))
    else:
        print("NOT FOUND")


floodevents_indofloods.csv
FOUND
Rows and columns: (4548, 13)
Column names: ['EventID', 'Start Date', 'End Date', 'Peak Flood Level (m)', 'Peak FL Date', 'Num Peak FL', 'Peak Discharge Q (cumec)', 'Peak Discharge Date', 'Flood Volume (cumec)', 'Event Duration (days)', 'Time to Peak (days)', 'Recession Time (day)', 'Flood Type']


,EventID,Start Date,End Date,Peak Flood Level (m),Peak FL Date,Num Peak FL,Peak Discharge Q (cumec),Peak Discharge Date,Flood Volume (cumec),Event Duration (days),Time to Peak (days),Recession Time (day),Flood Type
0,INDOFLOODS-gauge-1010-1,2010-07-21,2010-07-21,47.95,2010-07-21,1,NaN,NaN,NaN,1,1,1,Flood
1,INDOFLOODS-gauge-1010-2,2016-07-23,2016-07-23,48.05,2016-07-23,1,NaN,NaN,NaN,1,1,1,Flood
2,INDOFLOODS-gauge-1010-3,2016-07-26,2016-07-26,48.00,2016-07-26,1,NaN,NaN,NaN,1,1,1,Flood



metadata_indofloods.csv
FOUND
Rows and columns: (214, 18)
Column names: ['GaugeID', 'Warning Level', 'Danger Level', 'Station', 'Latitude', 'Longitude', 'River Name/ Tributory/ SubTributory', 'Basin', 'State', 'Start_date', 'End_date', 'Level_Entries', 'Streamflow_Entries', 'Privacy', 'Source Catchment Area', 'Catchment Area', 'Area variation (%)', 'Reliability']


,GaugeID,Warning Level,Danger Level,Station,Latitude,Longitude,River Name/ Tributory/ SubTributory,Basin,State,Start_date,End_date,Level_Entries,Streamflow_Entries,Privacy,Source Catchment Area,Catchment Area,Area variation (%),Reliability
0,INDOFLOODS-gauge-394,259.20,262.00,Biligundulu,12.1800,77.7300,Cauvery,Cauvery,Karnataka,1971-08-30,2020-05-27,17715,17273,Open,36682.0,36889.800780,0.566493,Safe
1,INDOFLOODS-gauge-118,66.75,67.75,Jainagar,26.5833,86.1481,Ganga/Kosi/ Kamla-Balan,Ganga - Brahmaputra - Meghna/Barak,Bihar,1970-07-01,2010-12-31,14134,13725,Restricted,2131.0,2118.569336,0.583325,Safe
2,INDOFLOODS-gauge-119,49.00,50.00,Jhanjharpur,26.2333,86.2594,Ganga/Kosi/ Kamla-Balan,Ganga - Brahmaputra - Meghna/Barak,Bihar,1970-01-01,2010-12-31,14939,14616,Restricted,2945.0,3504.423584,18.995707,Safe



precipitation_variables_indofloods.csv
FOUND
Rows and columns: (4548, 11)
Column names: ['EventID', 'T1d', 'T2d', 'T3d', 'T4d', 'T5d', 'T6d', 'T7d', 'T8d', 'T9d', 'T10d']


,EventID,T1d,T2d,T3d,T4d,T5d,T6d,T7d,T8d,T9d,T10d
0,INDOFLOODS-gauge-1010-1,32.833274,65.228899,99.413481,124.272676,133.344557,138.143367,140.458797,155.228323,177.372062,203.686949
1,INDOFLOODS-gauge-1010-2,27.011286,60.147585,86.528438,111.753411,136.083522,158.516418,197.476361,216.301547,243.307830,247.079665
2,INDOFLOODS-gauge-1010-3,50.413312,82.466743,124.859376,151.870662,185.006961,211.387814,236.612787,260.942897,283.375793,322.335737



catchment_characteristics_indofloods.csv
FOUND
Rows and columns: (155, 108)
Column names: ['GaugeID', 'Stream Order', 'Maximal Flow Length', 'Downvalley Length', 'Drainage Area', 'Catchment Relief', 'Catchment Length', 'Catchment Perimeter', 'Sinuosity Index', 'Form Factor', 'Relief Ratio', 'Elongation Ratio', 'Circularity Ratio', 'Lemniscates Value', 'Drainage Texture', 'Drainage Density', 'Compactness Coefficient', 'Wandering Ratio', 'Fitness Ratio', 'Basin Magnitude', 'Channel Frequency', 'Drainage Intensity', 'Infiltration Number', 'Ruggedness Number', 'No. of Firstorder Streams', 'No. of Secondorder Streams', 'No. of Thirdorder Streams', 'No. of Fourthorder Streams', 'No. of Fifthorder Streams', 'No. of Sixthorder Streams', 'No. of Seventhorder Streams', 'No. of Eigthorder Streams', 'Firstorder Streams Length', 'Secondorder Streams Length', 'Thirdorder Streams Length', 'Fourthorder Streams Length', 'Fifthorder Streams Length', 'Sixthorder Streams Length', 'Seventhorder Streams Le

,GaugeID,Stream Order,Maximal Flow Length,Downvalley Length,Drainage Area,Catchment Relief,Catchment Length,Catchment Perimeter,Sinuosity Index,Form Factor,...,2015_HDI,2005_HDI,2000_HDI,1995_HDI,1990_HDI,Population Count,Population Density,Land cover,Soil type,lithology type
0,INDOFLOODS-gauge-1013,4,141541.1719,113605.7734,3542.106445,4416,116870.2188,3.152483e+05,1.245898,0.259331,...,0.648626,0.567890,0.531529,0.500729,0.472248,1656813.0,465.074596,No dominant class,Cambisols,Unconsolidated sediments
1,INDOFLOODS-gauge-1018,5,277715.5938,183743.9063,9908.034180,7735,201079.8125,6.079071e+05,1.511428,0.245048,...,0.638331,0.548809,0.505907,0.471120,0.438265,2462084.0,247.254769,No dominant class,No dominant class,Metamorphics
2,INDOFLOODS-gauge-1020,5,463678.7188,330162.8125,21904.482420,1204,341410.3438,1.098149e+06,1.404394,0.187923,...,0.547139,0.469978,0.433152,0.403340,0.375281,15777280.0,722.637540,Cropland,Luvisols,Metamorphics


In [147]:
import os
import pandas as pd

folder = "/content/drive/MyDrive/MultiHazardIndia/raw_data/flood"

files = [
    "metadata_indofloods.csv",
    "precipitation_variables_indofloods.csv",
    "catchment_characteristics_indofloods.csv"
]

for filename in files:
    path = os.path.join(folder, filename)

    print("\n" + "="*65)
    print("FILE:", filename)

    if os.path.exists(path):
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        display(df.head(3))
    else:
        print("FILE NOT FOUND")


FILE: metadata_indofloods.csv
Rows: 214
Columns: ['GaugeID', 'Warning Level', 'Danger Level', 'Station', 'Latitude', 'Longitude', 'River Name/ Tributory/ SubTributory', 'Basin', 'State', 'Start_date', 'End_date', 'Level_Entries', 'Streamflow_Entries', 'Privacy', 'Source Catchment Area', 'Catchment Area', 'Area variation (%)', 'Reliability']


,GaugeID,Warning Level,Danger Level,Station,Latitude,Longitude,River Name/ Tributory/ SubTributory,Basin,State,Start_date,End_date,Level_Entries,Streamflow_Entries,Privacy,Source Catchment Area,Catchment Area,Area variation (%),Reliability
0,INDOFLOODS-gauge-394,259.20,262.00,Biligundulu,12.1800,77.7300,Cauvery,Cauvery,Karnataka,1971-08-30,2020-05-27,17715,17273,Open,36682.0,36889.800780,0.566493,Safe
1,INDOFLOODS-gauge-118,66.75,67.75,Jainagar,26.5833,86.1481,Ganga/Kosi/ Kamla-Balan,Ganga - Brahmaputra - Meghna/Barak,Bihar,1970-07-01,2010-12-31,14134,13725,Restricted,2131.0,2118.569336,0.583325,Safe
2,INDOFLOODS-gauge-119,49.00,50.00,Jhanjharpur,26.2333,86.2594,Ganga/Kosi/ Kamla-Balan,Ganga - Brahmaputra - Meghna/Barak,Bihar,1970-01-01,2010-12-31,14939,14616,Restricted,2945.0,3504.423584,18.995707,Safe



FILE: precipitation_variables_indofloods.csv
Rows: 4548
Columns: ['EventID', 'T1d', 'T2d', 'T3d', 'T4d', 'T5d', 'T6d', 'T7d', 'T8d', 'T9d', 'T10d']


,EventID,T1d,T2d,T3d,T4d,T5d,T6d,T7d,T8d,T9d,T10d
0,INDOFLOODS-gauge-1010-1,32.833274,65.228899,99.413481,124.272676,133.344557,138.143367,140.458797,155.228323,177.372062,203.686949
1,INDOFLOODS-gauge-1010-2,27.011286,60.147585,86.528438,111.753411,136.083522,158.516418,197.476361,216.301547,243.307830,247.079665
2,INDOFLOODS-gauge-1010-3,50.413312,82.466743,124.859376,151.870662,185.006961,211.387814,236.612787,260.942897,283.375793,322.335737



FILE: catchment_characteristics_indofloods.csv
Rows: 155
Columns: ['GaugeID', 'Stream Order', 'Maximal Flow Length', 'Downvalley Length', 'Drainage Area', 'Catchment Relief', 'Catchment Length', 'Catchment Perimeter', 'Sinuosity Index', 'Form Factor', 'Relief Ratio', 'Elongation Ratio', 'Circularity Ratio', 'Lemniscates Value', 'Drainage Texture', 'Drainage Density', 'Compactness Coefficient', 'Wandering Ratio', 'Fitness Ratio', 'Basin Magnitude', 'Channel Frequency', 'Drainage Intensity', 'Infiltration Number', 'Ruggedness Number', 'No. of Firstorder Streams', 'No. of Secondorder Streams', 'No. of Thirdorder Streams', 'No. of Fourthorder Streams', 'No. of Fifthorder Streams', 'No. of Sixthorder Streams', 'No. of Seventhorder Streams', 'No. of Eigthorder Streams', 'Firstorder Streams Length', 'Secondorder Streams Length', 'Thirdorder Streams Length', 'Fourthorder Streams Length', 'Fifthorder Streams Length', 'Sixthorder Streams Length', 'Seventhorder Streams Length', 'Eighthorder Stre

,GaugeID,Stream Order,Maximal Flow Length,Downvalley Length,Drainage Area,Catchment Relief,Catchment Length,Catchment Perimeter,Sinuosity Index,Form Factor,...,2015_HDI,2005_HDI,2000_HDI,1995_HDI,1990_HDI,Population Count,Population Density,Land cover,Soil type,lithology type
0,INDOFLOODS-gauge-1013,4,141541.1719,113605.7734,3542.106445,4416,116870.2188,3.152483e+05,1.245898,0.259331,...,0.648626,0.567890,0.531529,0.500729,0.472248,1656813.0,465.074596,No dominant class,Cambisols,Unconsolidated sediments
1,INDOFLOODS-gauge-1018,5,277715.5938,183743.9063,9908.034180,7735,201079.8125,6.079071e+05,1.511428,0.245048,...,0.638331,0.548809,0.505907,0.471120,0.438265,2462084.0,247.254769,No dominant class,No dominant class,Metamorphics
2,INDOFLOODS-gauge-1020,5,463678.7188,330162.8125,21904.482420,1204,341410.3438,1.098149e+06,1.404394,0.187923,...,0.547139,0.469978,0.433152,0.403340,0.375281,15777280.0,722.637540,Cropland,Luvisols,Metamorphics
